# Quote-Screening Signal Validation

This notebook checks whether the one-step-ahead predictive prices contain directional information about subsequent quote movements. It is not an executable trading backtest. The question is weaker and cleaner: when the model says a quote is cheap or expensive relative to predicted fair value, does the next observed mid quote tend to move in that direction?

For each contract observed at timestamp `t_k`, define the fair-value gap

`signal = predictive_price_mean - current_mid`.

The next quote movement is

`future_mid_change = next_mid - current_mid`.

The signal is directionally correct when these two quantities have the same sign.

In [ ]:
from pathlib import Path
import math
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_project_root(start=None):
    here = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError('Could not locate the project root containing src/ and notebooks/.')


def project_relative(path):
    path = Path(path)
    try:
        return path.relative_to(PROJECT_ROOT)
    except ValueError:
        return path


PROJECT_ROOT = find_project_root()
RUN_ROOT = PROJECT_ROOT / 'outputs' / 'common_eval_q160_eval300'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'comparisons' / 'quote_screening_signal_validation'
FIGURE_DIR = OUTPUT_DIR / 'figures'
TABLE_DIR = OUTPUT_DIR / 'tables'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

NORMAL_GLOB = 'normal_400ts_q160_eval300_seed*'
ROUGH_GLOB = 'rough_400ts_q160_eval300_seed*'
PREDICTION_FILE = 'predictive_option_prices_eval.csv'

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 140)

## Load Predictive Quote Files

The notebook uses the common-evaluation q160/q300 runs. The filters are updated on the cap-160 panel, while predictive diagnostics are evaluated on the broader q300 quote set. For this diagnostic, a quote is kept only if the same instrument has a valid bid and ask at both `t_k` and `t_{k+1}`.

In [ ]:
USECOLS = [
    'capture_time_utc',
    't_index',
    'instrument_name',
    'expiry',
    'time_to_expiry_years',
    'strike',
    'option_type_clean',
    'bid_price',
    'ask_price',
    'mid_price',
    'market_price',
    'log_moneyness',
    'predictive_price_mean',
    'predictive_price_std',
    'predictive_price_q05',
    'predictive_price_q50',
    'predictive_price_q95',
]

NUMERIC_COLS = [
    't_index',
    'time_to_expiry_years',
    'strike',
    'bid_price',
    'ask_price',
    'mid_price',
    'market_price',
    'log_moneyness',
    'predictive_price_mean',
    'predictive_price_std',
    'predictive_price_q05',
    'predictive_price_q50',
    'predictive_price_q95',
]


def parse_seed(path):
    match = re.search(r'seed(\d+)', path.name)
    if match is None:
        raise ValueError(f'Could not parse seed from {path.name}')
    return int(match.group(1))


def load_one_run(path, model):
    df = pd.read_csv(path / PREDICTION_FILE, usecols=lambda c: c in USECOLS)
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df['model'] = model
    df['seed'] = parse_seed(path)
    df['source_run'] = path.name
    df['t_index'] = df['t_index'].astype(int)
    df['current_mid'] = 0.5 * (df['bid_price'] + df['ask_price'])
    df['spread'] = df['ask_price'] - df['bid_price']
    df['relative_spread'] = df['spread'] / df['current_mid'].abs().replace(0.0, np.nan)
    df['valid_current_quote'] = (
        np.isfinite(df['bid_price'])
        & np.isfinite(df['ask_price'])
        & (df['ask_price'] >= df['bid_price'])
        & np.isfinite(df['current_mid'])
        & (df['current_mid'] > 0.0)
    )
    return df.sort_values(['t_index', 'instrument_name']).reset_index(drop=True)


def prepare_current_next(df):
    base = df[df['valid_current_quote']].copy()
    next_quotes = base[[
        't_index',
        'instrument_name',
        'current_mid',
        'bid_price',
        'ask_price',
        'spread',
    ]].copy()
    next_quotes['t_index'] = next_quotes['t_index'] - 1
    next_quotes = next_quotes.rename(columns={
        'current_mid': 'next_mid',
        'bid_price': 'next_bid_price',
        'ask_price': 'next_ask_price',
        'spread': 'next_spread',
    })
    paired = base.merge(
        next_quotes,
        on=['t_index', 'instrument_name'],
        how='inner',
        validate='one_to_one',
    )
    paired['future_mid_change'] = paired['next_mid'] - paired['current_mid']
    paired['future_mid_return'] = paired['future_mid_change'] / paired['current_mid']
    paired['signal'] = paired['predictive_price_mean'] - paired['current_mid']
    paired['signal_q50'] = paired['predictive_price_q50'] - paired['current_mid']
    paired['signal_over_spread'] = paired['signal'] / paired['spread'].replace(0.0, np.nan)
    paired['abs_signal_over_spread'] = paired['signal_over_spread'].abs()
    paired['signal_over_pred_sd'] = paired['signal'] / paired['predictive_price_std'].replace(0.0, np.nan)
    paired['signal_sign'] = np.sign(paired['signal'])
    paired['future_change_sign'] = np.sign(paired['future_mid_change'])
    paired['direction_defined'] = (paired['signal_sign'] != 0.0) & (paired['future_change_sign'] != 0.0)
    paired['direction_correct'] = (
        paired['direction_defined']
        & ((paired['signal_sign'] * paired['future_change_sign']) > 0.0)
    )
    paired['signed_mid_change'] = paired['signal_sign'] * paired['future_mid_change']
    paired['signed_mid_return'] = paired['signal_sign'] * paired['future_mid_return']
    paired['expiry_days'] = 365.0 * paired['time_to_expiry_years']
    return paired


frames = []
for model, pattern in [('normal', NORMAL_GLOB), ('rough', ROUGH_GLOB)]:
    paths = sorted(RUN_ROOT.glob(pattern))
    if not paths:
        raise FileNotFoundError(f'No run folders matched {RUN_ROOT / pattern}')
    for path in paths:
        print(f'Loading {model}: {path.name}')
        frames.append(prepare_current_next(load_one_run(path, model)))

paired = pd.concat(frames, ignore_index=True)
paired = paired[np.isfinite(paired['signal']) & np.isfinite(paired['future_mid_change'])].copy()
paired['abs_signal_percentile'] = paired.groupby(['model', 'seed'])['abs_signal_over_spread'].rank(pct=True)
paired.shape

In [ ]:
sample_summary = (
    paired.groupby(['model', 'seed'], as_index=False)
    .agg(
        n_quote_pairs=('instrument_name', 'size'),
        n_timestamps=('t_index', 'nunique'),
        n_contracts=('instrument_name', 'nunique'),
        mean_quotes_per_timestamp=('instrument_name', lambda x: x.size / paired.loc[x.index, 't_index'].nunique()),
        median_relative_spread=('relative_spread', 'median'),
        median_expiry_days=('expiry_days', 'median'),
    )
)
sample_summary.to_csv(TABLE_DIR / 'quote_screening_sample_summary.csv', index=False)
sample_summary

## Descriptive Signal Tests

The main quote-level diagnostic is the directional hit rate. A value above 50 percent means that, more often than not, the next mid quote moved in the direction implied by the model's fair-value gap. Since quote-level observations are highly dependent within a timestamp, these quote-level summaries are descriptive. Formal paired inference is done later at timestamp level.

In [ ]:
def summarize_quote_level(df, label):
    rows = []
    for (model, seed), g in df.groupby(['model', 'seed'], sort=True):
        defined = g[g['direction_defined']].copy()
        rows.append({
            'sample': label,
            'model': model,
            'seed': seed,
            'n_quote_pairs': len(g),
            'n_direction_defined': len(defined),
            'hit_rate': defined['direction_correct'].mean() if len(defined) else np.nan,
            'mean_signed_mid_change': defined['signed_mid_change'].mean() if len(defined) else np.nan,
            'median_signed_mid_change': defined['signed_mid_change'].median() if len(defined) else np.nan,
            'mean_signed_mid_return': defined['signed_mid_return'].mean() if len(defined) else np.nan,
            'mean_abs_signal_over_spread': g['abs_signal_over_spread'].mean(),
            'corr_signal_future_change': g[['signal', 'future_mid_change']].corr().iloc[0, 1],
            'spearman_signal_future_change': g[['signal', 'future_mid_change']].corr(method='spearman').iloc[0, 1],
        })
    out = pd.DataFrame(rows)
    pooled = (
        out.groupby(['sample', 'model'], as_index=False)
        .agg(
            seed=('seed', lambda _: 'pooled_mean'),
            n_quote_pairs=('n_quote_pairs', 'mean'),
            n_direction_defined=('n_direction_defined', 'mean'),
            hit_rate=('hit_rate', 'mean'),
            mean_signed_mid_change=('mean_signed_mid_change', 'mean'),
            median_signed_mid_change=('median_signed_mid_change', 'mean'),
            mean_signed_mid_return=('mean_signed_mid_return', 'mean'),
            mean_abs_signal_over_spread=('mean_abs_signal_over_spread', 'mean'),
            corr_signal_future_change=('corr_signal_future_change', 'mean'),
            spearman_signal_future_change=('spearman_signal_future_change', 'mean'),
        )
    )
    return pd.concat([out, pooled], ignore_index=True)


samples = {
    'all_valid_quote_pairs': paired,
    'relative_spread_le_20pct': paired[paired['relative_spread'] <= 0.20],
    'relative_spread_le_15pct': paired[paired['relative_spread'] <= 0.15],
    'top_20pct_abs_signal_over_spread': paired[paired['abs_signal_percentile'] >= 0.80],
    'top_10pct_abs_signal_over_spread': paired[paired['abs_signal_percentile'] >= 0.90],
    'rel_spread_le_15pct_and_top_20pct_signal': paired[(paired['relative_spread'] <= 0.15) & (paired['abs_signal_percentile'] >= 0.80)],
}

quote_summaries = pd.concat(
    [summarize_quote_level(df, label) for label, df in samples.items()],
    ignore_index=True,
)
quote_summaries.to_csv(TABLE_DIR / 'quote_level_signal_validation_summary.csv', index=False)
quote_summaries[quote_summaries['seed'].astype(str).eq('pooled_mean')]

## Timestamp-Level Paired Inference

The formal comparison treats each timestamp as the observational unit. For each seed and timestamp, quote-level outcomes are averaged inside the panel. Rough-minus-Normal differences are then averaged across seeds at the same timestamp, and HAC standard errors are used for the mean timestamp difference.

In [ ]:
def default_hac_lag(n):
    return max(1, int(math.floor(4.0 * (n / 100.0) ** (2.0 / 9.0))))


def hac_mean_test(values, lag=None):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    if n == 0:
        return {'n_timestamps': 0, 'mean': np.nan, 'hac_lag': np.nan, 'hac_se': np.nan, 't_stat': np.nan, 'p_value_normal': np.nan, 'ci95_lower': np.nan, 'ci95_upper': np.nan}
    use_lag = default_hac_lag(n) if lag is None else int(lag)
    use_lag = min(use_lag, n - 1)
    mean = float(np.mean(x))
    centered = x - mean
    gamma0 = float(np.dot(centered, centered) / n)
    long_run_var = gamma0
    for ell in range(1, use_lag + 1):
        gamma = float(np.dot(centered[ell:], centered[:-ell]) / n)
        weight = 1.0 - ell / (use_lag + 1.0)
        long_run_var += 2.0 * weight * gamma
    long_run_var = max(long_run_var, 0.0)
    hac_se = math.sqrt(long_run_var / n)
    t_stat = mean / hac_se if hac_se > 0 else np.nan
    p_value = math.erfc(abs(t_stat) / math.sqrt(2.0)) if np.isfinite(t_stat) else np.nan
    ci_half = 1.96 * hac_se
    return {'n_timestamps': n, 'mean': mean, 'hac_lag': use_lag, 'hac_se': hac_se, 't_stat': t_stat, 'p_value_normal': p_value, 'ci95_lower': mean - ci_half, 'ci95_upper': mean + ci_half}


def timestamp_metrics(df):
    defined = df[df['direction_defined']].copy()
    return (
        defined.groupby(['model', 'seed', 't_index'], as_index=False)
        .agg(
            n_quotes=('instrument_name', 'size'),
            hit_rate=('direction_correct', 'mean'),
            mean_signed_mid_change=('signed_mid_change', 'mean'),
            median_signed_mid_change=('signed_mid_change', 'median'),
            mean_signed_mid_return=('signed_mid_return', 'mean'),
            mean_abs_signal_over_spread=('abs_signal_over_spread', 'mean'),
        )
    )


def rough_minus_normal_hac(df, sample_label):
    tm = timestamp_metrics(df)
    metrics = ['hit_rate', 'mean_signed_mid_change', 'median_signed_mid_change', 'mean_signed_mid_return']
    rows = []
    for metric in metrics:
        pivot = tm.pivot_table(index=['seed', 't_index'], columns='model', values=metric)
        pivot = pivot.dropna(subset=['rough', 'normal']).reset_index()
        pivot['rough_minus_normal'] = pivot['rough'] - pivot['normal']
        seed_avg = pivot.groupby('t_index', as_index=False)['rough_minus_normal'].mean()
        stats = hac_mean_test(seed_avg['rough_minus_normal'].to_numpy())
        rows.append({
            'sample': sample_label,
            'metric': metric,
            **stats,
        })
    return pd.DataFrame(rows)


hac_results = pd.concat(
    [rough_minus_normal_hac(df, label) for label, df in samples.items()],
    ignore_index=True,
)
hac_results.to_csv(TABLE_DIR / 'timestamp_level_rough_minus_normal_hac.csv', index=False)
hac_results

## Plots

The first plot bins quotes by the model signal scaled by the bid-ask spread. If the signal is directionally informative, average future mid changes should increase across the bins. The second plot reports directional hit rates for stronger signal subsets.

In [ ]:
def signal_decile_table(df):
    pieces = []
    for model, g in df.groupby('model', sort=True):
        h = g[np.isfinite(g['signal_over_spread']) & np.isfinite(g['future_mid_change'])].copy()
        h['signal_decile'] = pd.qcut(h['signal_over_spread'], q=10, labels=False, duplicates='drop')
        pieces.append(
            h.groupby('signal_decile', as_index=False)
            .agg(
                model=('model', 'first'),
                n_quotes=('instrument_name', 'size'),
                mean_signal_over_spread=('signal_over_spread', 'mean'),
                mean_future_mid_change=('future_mid_change', 'mean'),
                mean_future_mid_return=('future_mid_return', 'mean'),
                hit_rate=('direction_correct', 'mean'),
            )
        )
    return pd.concat(pieces, ignore_index=True)


deciles = signal_decile_table(paired)
deciles.to_csv(TABLE_DIR / 'signal_decile_future_move_table.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
colors = {'normal': '#6b7280', 'rough': '#2563eb'}
for model, g in deciles.groupby('model', sort=True):
    axes[0].plot(g['mean_signal_over_spread'], g['mean_future_mid_change'], marker='o', label=model, color=colors.get(model))
    axes[1].plot(g['mean_signal_over_spread'], g['hit_rate'], marker='o', label=model, color=colors.get(model))
axes[0].axhline(0, color='black', linewidth=1)
axes[0].axvline(0, color='black', linewidth=1)
axes[0].set_title('Future mid move by signal decile')
axes[0].set_xlabel('mean signal / spread')
axes[0].set_ylabel('mean next mid change, BTC')
axes[1].axhline(0.5, color='black', linewidth=1, linestyle='--')
axes[1].set_title('Directional hit rate by signal decile')
axes[1].set_xlabel('mean signal / spread')
axes[1].set_ylabel('hit rate')
axes[1].legend(frameon=False)
fig.savefig(FIGURE_DIR / 'signal_decile_future_move.png', dpi=180)
plt.show()

In [ ]:
plot_summary = quote_summaries[
    quote_summaries['seed'].astype(str).eq('pooled_mean')
    & quote_summaries['sample'].isin([
        'all_valid_quote_pairs',
        'relative_spread_le_15pct',
        'top_20pct_abs_signal_over_spread',
        'top_10pct_abs_signal_over_spread',
        'rel_spread_le_15pct_and_top_20pct_signal',
    ])
].copy()

fig, ax = plt.subplots(figsize=(10, 4.5))
labels = list(plot_summary['sample'].drop_duplicates())
x = np.arange(len(labels))
width = 0.36
for offset, model in [(-width / 2, 'normal'), (width / 2, 'rough')]:
    g = plot_summary[plot_summary['model'].eq(model)].set_index('sample').reindex(labels)
    ax.bar(x + offset, g['hit_rate'], width=width, label=model, color=colors.get(model))
ax.axhline(0.5, color='black', linestyle='--', linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=25, ha='right')
ax.set_ylabel('directional hit rate')
ax.set_title('Quote-screening directional hit rates')
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'directional_hit_rates_by_signal_subset.png', dpi=180)
plt.show()

## Reading The Output

A useful positive result would look like this: hit rates above 50 percent, positive mean signed mid changes, and Rough-SABR improving over Normal-SABR at timestamp level. A weak result would mean that the predictive-score advantage does not translate into clear directional quote revisions. Either outcome is still useful, because this diagnostic connects predictive pricing to subsequent market movement without imposing a full market-taking trading strategy.